In [ ]:
import os
import numpy as np
import pandas as pd

from _plot_utils import plot_radar, plot_roc, plot_lift
from pass_pclr.defines import ECHONEXT_TARGETS, ECHONEXT_COMPOSITE_TARGET


def _get_task_row(df, task, metrics_file):
    task_data = df[df["Label"] == task]
    if len(task_data) == 0:
        raise ValueError(f"Task {task} missing in {metrics_file}")
    elif len(task_data) > 1:
        raise ValueError(f"Task {task} has more than one entry in {metrics_file}")
    return task_data.iloc[0]


# Function to load all experiment data
def load_experiment_data(runs_dir="runs/", composite_idx=-1):
    """Load metrics from all experiment directories"""
    experiments = {}

    # Get all experiment directories
    for exp_dir in sorted(os.listdir(runs_dir)):
        exp_path = os.path.join(runs_dir, exp_dir)

        # Check if it's a directory
        if os.path.isdir(exp_path):
            metrics_file = os.path.join(exp_path, "metrics.csv")
            probs_file = os.path.join(exp_path, "probs.npy")

            # Check if metrics.csv exists
            if os.path.exists(metrics_file):
                df = pd.read_csv(metrics_file)

                # Get AUROC values for multilabel tasks
                multilabel_aurocs = []
                multilabel_auprcs = []
                for task in ECHONEXT_TARGETS:
                    if task == ECHONEXT_COMPOSITE_TARGET:
                        continue
                    task_data = _get_task_row(df, task, metrics_file)
                    multilabel_aurocs.append(task_data["AUROC"])
                    multilabel_auprcs.append(task_data["AUPRC"])

                composite_data = _get_task_row(df, "SHD", metrics_file)
                multilabel_avg_data = _get_task_row(
                    df, "Multilabel Averaged", metrics_file
                )

                experiments[exp_dir] = {
                    "multilabel_aurocs": multilabel_aurocs,
                    "multilabel_avg_auroc": multilabel_avg_data["AUROC"],
                    "composite_auroc": composite_data["AUROC"],
                    "multilabel_avg_auprc": multilabel_avg_data["AUPRC"],
                    "composite_auprc": composite_data["AUPRC"],
                    "all_data": df,
                }

            # Check if probs.npy exists
            if os.path.exists(probs_file):
                probs = np.load(probs_file, allow_pickle=True)

                experiments[exp_dir]["y_prob"] = probs[:, composite_idx]

    return experiments

In [ ]:
df_meta = pd.read_csv("/opt/gpudata/ecg/echonext/EchoNext_metadata_100k.csv")
y_true = df_meta.loc[df_meta["split"] == "test", "shd_moderate_or_greater_flag"].to_numpy()

experiments = {
    "full": load_experiment_data("../outputs/runs-echonext/"),
    "32k": load_experiment_data("../outputs/runs-echonext-32k/"),
    "16k": load_experiment_data("../outputs/runs-echonext-16k/"),
    "8k": load_experiment_data("../outputs/runs-echonext-8k/"),
    "4k": load_experiment_data("../outputs/runs-echonext-4k/"),
    "2k": load_experiment_data("../outputs/runs-echonext-2k/"),
    "1k": load_experiment_data("../outputs/runs-echonext-1k/"),
    "512": load_experiment_data("../outputs/runs-echonext-512/"),
    "256": load_experiment_data("../outputs/runs-echonext-256/"),
}

print("Experiments found:")
for exp_subset, subset_results in experiments.items():
    print(f"\tSubset {exp_subset}:")
    for exp_name, exp_data in subset_results.items():
        print(f"\t\t{exp_name}: composite AUROC = {exp_data['composite_auroc']:.4f}")

In [ ]:
baselines_full = {a: (experiments[s][k], c, l, m) for a, s, k, c, l, m in [
    # ------------------------------------------------------------------------------------------------
    #        Alias            Subset        Key                         Color       Line    Marker
    # ------------------------------------------------------------------------------------------------
    ("columbia-minimodel",    "full", "columbia-minimodel",          "black",       "-",    None),
    ("tabular-logreg",        "full", "tabular-logreg-unweighted",   "tab:red",     "-",    None),
    ("proto-from-scratch",    "full", "proto-from-scratch",       "tab:brown",   "-",    None),
    # ("proto-ptbxl-pip",       "full", "proto-ptbxl-pip-cat3",        "tab:green",   "-",    None),
    # ("proto-ptbxl-pit",       "full", "proto-ptbxl-pit-cat3",        "tab:pink",    "-",    None),
    # ("pass-ptbxl-pip",        "full", "pass-ptbxl-pip",              "tab:olive",   "-",    None),
    # ("pass-ptbxl-pit",        "full", "pass-ptbxl-pit",              "tab:cyan",    "-",    None),
    ("pass-heedb-pip",        "full", "pass-heedb-pip",              "tab:blue",    "-",    None),
    ("pass-heedb-pit",        "full", "pass-heedb-pit",              "tab:orange",  "-",    None),
    ("pass-heedb-pip-logreg", "full", "pass-heedb-pip-logreg",       "tab:purple",  "-",    None),
    ("pass-heedb-pit-logreg", "full", "pass-heedb-pit-logreg",       "tab:gray",    "-",    None),
    # ------------------------------------------------------------------------------------------------
]}

In [ ]:
plot_radar(baselines_full, labels=[x for x in ECHONEXT_TARGETS if x != "SHD"], title="Full-data Multilabel AUROCs", save_path="figs/echonext-radar-full.png")

In [ ]:
plot_roc(baselines_full, y_true, title="Full-data SHD ROC Curves", save_path="figs/echonext-roc-full.png")

In [ ]:
FULL_SIZE = 72475

sizes = {
    "full": 72475,
    "32k": 32768,
    "16k": 16384,
    "8k": 8192,
    "4k": 4096,
    "2k": 2048,
    "1k": 1024,
    "512": 512,
    "256": 256,
}

aliases = {
    "columbia-minimodel": "columbia-minimodel",
    "tabular-logreg-unweighted": "tabular-logreg",
    # "proto-from-scratch": "proto-from-scratch",
    "proto-from-scratch-logreg": "proto-from-scratch-logreg",
    "proto-ptbxl-pip-fusion": "proto-ptbxl-pip",
    "proto-ptbxl-pit-fusion": "proto-ptbxl-pit",
    # "pass-ptbxl-pip": "pass-ptbxl-pip",
    # "pass-ptbxl-pit": "pass-ptbxl-pit",
    # "pass-heedb-pip": "pass-heedb-pip",
    # "pass-heedb-pit": "pass-heedb-pit",
    "pass-heedb-pip-logreg": "pass-heedb-pip-logreg",
    "pass-heedb-pit-logreg": "pass-heedb-pit-logreg",
    # "pass-heedb-pit-assign": "pass-heedb-pit-assign",
    "pass-heedb-pit-assign-logreg": "pass-heedb-pit-assign-logreg",
}

palette = {
    "columbia-minimodel": "black",
    "tabular-logreg": "tab:red",
    # "proto-from-scratch": "tab:brown",
    "proto-from-scratch-logreg": "tab:green",
    "proto-ptbxl-pip": "tab:purple",
    "proto-ptbxl-pit": "tab:gray",
    # "pass-ptbxl-pip": "tab:olive",
    # "pass-ptbxl-pit": "tab:cyan",
    # "pass-heedb-pip": "tab:blue",
    # "pass-heedb-pit": "tab:orange",
    "pass-heedb-pip-logreg": "tab:blue",
    "pass-heedb-pit-logreg": "tab:orange",
    # "pass-heedb-pit-assign": "tab:purple",
    "pass-heedb-pit-assign-logreg": "tab:purple",
}

df = pd.DataFrame.from_records(
    [
        {
            "Model": aliases[exp_name],
            "Train Size": sizes[exp_subset],
            "SHD (AUROC)": exp_data["composite_auroc"],
            "SHD (AUPRC)": exp_data["composite_auprc"],
            "Multilabel (AUROC)": exp_data["multilabel_avg_auroc"],
            "Multilabel (AUPRC)": exp_data["multilabel_avg_auprc"],
        }
        for exp_subset, subset_results in experiments.items()
        for exp_name, exp_data in subset_results.items()
        if exp_name in aliases
    ]
)

In [ ]:
df_main = df[df["Model"].isin(["columbia-minimodel", "tabular-logreg", "proto-from-scratch-logreg", "pass-heedb-pip-logreg", "pass-heedb-pit-logreg", "pass-heedb-pit-assign-logreg"])]

plot_lift(
    data=df_main,
    metric="SHD (AUROC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Composite (AUROC)",
    save_path="figs/echonext-composite-lift-roc.png",
    ylim=(0.5, 0.9),
)
plot_lift(
    data=df_main,
    metric="Multilabel (AUROC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Multilabel (AUROC)",
    save_path="figs/echonext-multilabel-lift-roc.png",
    ylim=(0.5, 0.9),
)
plot_lift(
    data=df_main,
    metric="SHD (AUPRC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Composite (AUPRC)",
    save_path="figs/echonext-composite-lift-pr.png",
    ylim=(0.5, 0.9),
)
plot_lift(
    data=df_main,
    metric="Multilabel (AUPRC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Multilabel (AUPRC)",
    save_path="figs/echonext-multilabel-lift-pr.png",
    ylim=(0.05, 0.35),
)

In [ ]:
df_main = df[df["Model"].isin(["columbia-minimodel", "tabular-logreg", "proto-from-scratch-logreg", "proto-ptbxl-pip", "proto-ptbxl-pit"])]

plot_lift(
    data=df_main,
    metric="SHD (AUROC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Composite (AUROC)",
    save_path="figs/protoecgnet-echonext-composite-lift-roc.png",
    ylim=(0.5, 0.9),
)
plot_lift(
    data=df_main,
    metric="Multilabel (AUROC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Multilabel (AUROC)",
    save_path="figs/protoecgnet-echonext-multilabel-lift-roc.png",
    ylim=(0.5, 0.9),
)
plot_lift(
    data=df_main,
    metric="SHD (AUPRC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Composite (AUPRC)",
    save_path="figs/protoecgnet-echonext-composite-lift-pr.png",
    ylim=(0.5, 0.9),
)
plot_lift(
    data=df_main,
    metric="Multilabel (AUPRC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="EchoNext Multilabel (AUPRC)",
    save_path="figs/protoecgnet-echonext-multilabel-lift-pr.png",
    ylim=(0.05, 0.35),
)